In [ ]:
!pip install -q kagglehub librosa
import kagglehub, glob, os, collections, librosa, numpy as np

In [ ]:
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
print("Downloaded to:", path)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Downloaded to: /kaggle/input/gtzan-dataset-music-genre-classification


In [ ]:
import glob, os, collections
data = sorted(glob.glob(os.path.join(path, "**", "*.wav"), recursive=True))
counts = collections.Counter(os.path.basename(os.path.dirname(i)) for i in data)
print(f"Found {len(data)} data files across {len(counts)} genres\n")
for genre, n in sorted(counts.items()):
    print(f"  {genre:10s} {n}")

Found 1000 data files across 10 genres

  blues      100
  classical  100
  country    100
  disco      100
  hiphop     100
  jazz       100
  metal      100
  pop        100
  reggae     100
  rock       100


In [ ]:
from sklearn.model_selection import train_test_split

genres = sorted(set(os.path.basename(os.path.dirname(i)) for i in data))
genre_index = {g: i for i, g in enumerate(genres)}

labels = [genre_index[os.path.basename(os.path.dirname(i))] for i in data]

train_data, temp_data, train_labels, temp_labels = train_test_split(
    data, labels, test_size=0.30, stratify=labels, random_state=42)
val_data, test_data, val_labels, test_labels = train_test_split(
    temp_data, temp_labels, test_size=0.50, stratify=temp_labels, random_state=42)

print(f"Train: {len(train_data)} songs")
print(f"Val:   {len(val_data)} songs")
print(f"Test:  {len(test_data)} songs")
print("Genre index map:", genre_index)

Train: 700 songs
Val:   150 songs
Test:  150 songs
Genre index map: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [ ]:
import torch
import numpy as np
import librosa
from torch.utils.data import Dataset, DataLoader

SAMPLE_RATE = 22050      # samples per second we resample every clip to
CHUNK_SECONDS = 5        # length of each chunk
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SECONDS
N_MELS = 128             # height of the spectrogram

class GTZANDataset(Dataset):
    def __init__(self, data, labels):
        self.index = []
        for filepath, label in zip(data, labels):
            try:
                duration = librosa.get_duration(path=filepath)
            except Exception:
                continue
            n_chunks = int(duration // CHUNK_SECONDS)
            for c in range(n_chunks):
                self.index.append((filepath, label, c))

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        filepath, label, chunk = self.index[i]

        offset = chunk * CHUNK_SECONDS
        y, sr = librosa.load(filepath, sr=SAMPLE_RATE,
                             offset=offset, duration=CHUNK_SECONDS)

        if len(y) < CHUNK_SAMPLES:
            y = np.pad(y, (0, CHUNK_SAMPLES - len(y)))

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

        tensor = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)
        return tensor, label

In [ ]:
train_ds = GTZANDataset(train_data, train_labels)
val_ds   = GTZANDataset(val_data,   val_labels)
test_ds  = GTZANDataset(test_data,  test_labels)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

print(f"Train chunks: {len(train_ds)}")
print(f"Val chunks:   {len(val_ds)}")
print(f"Test chunks:  {len(test_ds)}")

/tmp/ipykernel_6683/1633446368.py:16: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  duration = librosa.get_duration(path=filepath)


Train chunks: 4185
Val chunks:   900
Test chunks:  900


In [ ]:
import torch.nn as nn
class GenreCNN(nn.Module):
  def __init__(self, n_classes=10):
    super().__init__()

    self.conv = nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.MaxPool2d(2),

    )

    self.pool = nn.AdaptiveAvgPool2d(1)

    self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
    )

  def forward(self, x):
    x = self.conv(x)
    x = self.pool(x)
    x = self.head(x)
    return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GenreCNN(n_classes=10).to(device)
print("Device:", device)

# Push one real batch through to confirm the shapes line up.
x, y = next(iter(train_loader))
out = model(x.to(device))
print("Output shape:", out.shape)   # expect (32, 10)

Device: cuda
Output shape: torch.Size([32, 10])


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 20
best_val_acc = 0.0
history = {"train_loss": [], "train_acc": [], "val_acc": []}

def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            predicted = model(x).argmax(dim=1)
            correct += (predicted == y).sum().item()
            total += y.size(0)
    return correct / total

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        correct += (out.argmax(dim=1) == y).sum().item()
        total += y.size(0)


    train_loss = running_loss / total
    train_acc = correct / total
    val_acc = evaluate(val_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

    print(f"Epoch {epoch:2d} | train loss {train_loss:.3f} | "
          f"train acc {train_acc:.3f} | val acc {val_acc:.3f}")

print(f"\nBest validation accuracy: {best_val_acc:.3f}")


Epoch  1 | train loss 1.542 | train acc 0.460 | val acc 0.583
Epoch  2 | train loss 1.155 | train acc 0.610 | val acc 0.500
Epoch  3 | train loss 0.988 | train acc 0.667 | val acc 0.599
Epoch  4 | train loss 0.892 | train acc 0.703 | val acc 0.391
Epoch  5 | train loss 0.805 | train acc 0.732 | val acc 0.636
Epoch  6 | train loss 0.744 | train acc 0.755 | val acc 0.672
Epoch  7 | train loss 0.706 | train acc 0.763 | val acc 0.712
Epoch  8 | train loss 0.659 | train acc 0.786 | val acc 0.666
Epoch  9 | train loss 0.627 | train acc 0.791 | val acc 0.632
Epoch 10 | train loss 0.595 | train acc 0.805 | val acc 0.613
Epoch 11 | train loss 0.565 | train acc 0.811 | val acc 0.503
Epoch 12 | train loss 0.537 | train acc 0.821 | val acc 0.578
Epoch 13 | train loss 0.500 | train acc 0.840 | val acc 0.666
Epoch 14 | train loss 0.501 | train acc 0.835 | val acc 0.556
Epoch 15 | train loss 0.457 | train acc 0.852 | val acc 0.538
Epoch 16 | train loss 0.428 | train acc 0.858 | val acc 0.649
Epoch 17

In [ ]:
import os
print("Exists:", os.path.exists("best_model.pth"))

from google.colab import files
files.download("best_model.pth")

Exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from collections import Counter
from sklearn.metrics import accuracy_score

def evaluate_song_level(data, labels):
  model.eval()
  song_prediction = []
  song_actual = []

  with torch.no_grad():
    for filepath, true_label in zip(data, labels):
      d = GTZANDataset([filepath], [true_label])
      loader = DataLoader(d, batch_size=16, shuffle=False)

      chunk_predictions = []
      for x, _ in loader:
        x = x.to(device)
        predictions = model(x).argmax(dim=1)
        chunk_predictions.extend(predictions.cpu().tolist())

      if len(chunk_predictions) == 0:
        continue

      frequent = Counter(chunk_predictions).most_common(1)[0][0]
      song_prediction.append(frequent)
      song_actual.append(true_label)

    accuracy = accuracy_score(song_actual, song_prediction)
    return accuracy, song_actual, song_prediction

model.load_state_dict(torch.load("best_model.pth"))
song_accuracy, y_true, y_pred = evaluate_song_level(test_data, test_labels)
print(f"Song-level test accuracy: {song_accuracy:.3f}")


Song-level test accuracy: 0.827


In [ ]:
import os
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

model.eval()

def get_identity(x):
  feats = model.conv(x)
  feats = model.pool(feats)
  return feats.flatten(1)

def build_library(dataset):
  loader = DataLoader(dataset, batch_size=32, shuffle=False)
  all_identities = []
  with torch.no_grad():
    for x, _ in loader:
      all_identities.append(get_identity(x.to(device)).cpu())
  all_identities = torch.cat(all_identities).numpy()

  sort_song = defaultdict(list)
  for (filepath, _label, _chunk), identity in zip(dataset.index, all_identities):
    sort_song[filepath].append(identity)

  song_files = list(sort_song.keys())
  song_identity  = np.stack([np.mean(sort_song[f], axis=0) for f in song_files])
  return song_files, song_identity

all_dataset = GTZANDataset(data, labels)
song_files, song_identity = build_library(all_dataset)

def most_similar(index, k=5):
  similarity  = cosine_similarity(song_identity[index:index+1], song_identity)[0]
  order = [i for i in np.argsort(similarity)[::-1] if i != index][:k]

  name = os.path.basename(song_files[index])
  print(f"\nTop Similar Songs: {name}")
  print("-" * 40)
  for rank, i in enumerate(order, 1):
    name = os.path.basename(song_files[i])
    print(f"{rank}. {name:22s} similarity {similarity[i]:.3f}")


most_similar(0)



/tmp/ipykernel_6683/1633446368.py:16: FutureWarning: PySoundFile failed. Trying audioread instead.
	Audioread support is deprecated in librosa 0.10.0 and will be removed in version 1.0.
  duration = librosa.get_duration(path=filepath)



Top Similar Songs: blues.00000.wav
----------------------------------------
1. blues.00055.wav        similarity 0.984
2. blues.00050.wav        similarity 0.982
3. blues.00047.wav        similarity 0.979
4. blues.00069.wav        similarity 0.979
5. blues.00001.wav        similarity 0.979
